# TN2211 Session 8: Transistors

## Instruments


In [1]:
import sys
sys.path.append("../drivers/")
from tn2211_drivers import *
import glob
import matplotlib.pyplot as plt
import math
import numpy as np
import time
from scipy.optimize import curve_fit
from scipy.signal import hilbert

Loading BokehJS ...

In [2]:
import pyvisa
rm = pyvisa.ResourceManager()
rm.list_resources()

('ASRL1::INSTR',
 'ASRL2::INSTR',
 'USB0::0xF4EC::0x1103::SDG1XDDX802455::INSTR',
 'USB0::0xF4EC::0x1017::SDS08A0Q808388::INSTR')

In [3]:
scope = Scope("SDS")
gen = Generator("SDG")

Connecting to scope at resource: USB0::0xF4EC::0x1017::SDS08A0Q808388::INSTR
Setting memory depth to 10k points
Connecting to scope at resource: USB0::0xF4EC::0x1103::SDG1XDDX802455::INSTR


## Step 1: Shunt feedback amplifier

Set up the instruments. We will use triggering from the SYNC out of the generator which should be fed into Channel 4 of the scope.

In [4]:
# Set up the generators
gen.write("C1:BSWV WVTP,SINE,FRQ,1e3,AMP,0.1,OFST,0")
gen.write("C2:BSWV WVTP,DC,OFST,5")
gen.write("C1:OUTP ON")
gen.write("C2:OUTP ON")
gen.write("C1:SYNC ON,TYPE,CH1")

# Configure the channels we will use
scope.write("CHAN1:SWIT ON")
scope.write("CHAN2:SWIT ON")
scope.write("CHAN1:COUP DC")
scope.write("CHAN2:COUP DC")
scope.write("CHAN3:SWIT OFF")
scope.write("CHAN1:SCAL 0.1")
scope.write("CHAN2:SCAL 0.1")
scope.write("CHAN1:OFFS 0")
scope.write("CHAN2:OFFS 0")
scope.write("CHAN1:VIS ON")
scope.write("CHAN2:VIS ON")
scope.write("TIM:SCAL .1e-3")
scope.write("ACQ:MDEP 10k")

# We will use Ch4 of the scope connected to the sync out of the 
# generator for triggering
scope.write("CHAN4:SWIT ON")
scope.write("CHAN4:SCAL 2")
scope.write("TRIG:EDGE:SOUR C4")
scope.write("TRIG:EDGE:LEV 1")
scope.write("CHAN4:VIS OFF")
scope.write("TRIG:RUN")

These should be a reasonable set of starting points, you should change the scope ranges to make things more visible and to capture the information from the traces that you want to. 

For your logbook and Summary and Analysis (S&A) report:

In [ ]:
# For your logbook:
scope.get_screenshot()

Try adjusting the DC offset of the input signal and take some more screenshots of what happens for your logbook and describe what you see when you do this in your S&A report: 

In [ ]:
gen.set_offset(1,10e-3)

In [ ]:
gen.set_offset(1,-10e-3)

## Step 2: An AC-coupled shunt feedback amplifier with DC offset correction

You can use the same settings as above and the code above to take the screenshots.

## Step 3: Observing the gain for different feedback resistors

Set the scope channels to AC coupling:

In [ ]:
scope.write("CHAN1:COUP AC")
scope.write("CHAN2:COUP AC")

For higher gains, you will likely need to reduce the amplitude of your input signal to prevent your amplifier output from saturating (clipping):

In [ ]:
gen.set_amplitude(1, 10e-3) # 10 mV? You may need to reduce it even more. 

Get screenshots for your logbook:

In [ ]:
# For your logbook:
scope.get_screenshot()

To measure the gain, you can either:

* Use the built-in measurements buttons on the scope, and include those in your logbook, referring to them in your S&A report where you describre the gain calculation
* Use the code to download the traces, fit a cosine, and then calculate the gain. In this case, you can include screenshots of the code output and quote the gains in your S&A report

In [ ]:
def f(t,a,b,c):
    return a*np.cos(2*np.pi*1e3*t+b)+c

def get_fit(t,v):
    (a,b,c), cov = curve_fit(f,t,v)
    return a,b,c

def get_amplitude(t,v):
    return get_fit(t,v)[0]

# In general, it is best to stop the scope trigger before grabbing the traces 
scope.write("TRIG:STOP")
v_in = get_trace(1)
v_out = get_trace(2)
scope.write("TRIG:RUN")

amp_in = get_amplitude(t,v_in)
amp_out = get_amplitude(t,v_out)

print("Amplitude in:  %e V" % amp_in)
print("Amplitude out: %e V" % amp_out)
print("Voltage gain: %e" % (amp_out / amp_in)

## Step 4: Bandwidth of a shunt feedback amplifier

For this, we will configure a frequency sweep ("chirp") of the input signal, capture traces of both the input singal and output signal during the frequency sweep, and then use the magic of the [Hilbert transform](https://en.wikipedia.org/wiki/Hilbert_transform) to turn this into the frequency dependent amplification 

$$ 
A(\omega) = v_{out}(\omega) / v_{in}(\omega)
$$

The Hilbert transform will return a complex variable, containing both the amplitude of the gain, and the phase that the amplifier imparts on the signal. 

This is done with code we will reuse and improve from Session 5 where you studied resonance:

In [ ]:
# Set up the generators
gen.write("C1:BSWV WVTP,SINE")
gen.write("C2:BSWV WVTP,DC,OFST,5")
gen.write("C1:OUTP ON")
gen.write("C2:OUTP ON")
gen.write("C1:SYNC ON,TYPE,CH1")

# Configure the channels we will use
scope.write("CHAN1:SWIT ON")
scope.write("CHAN2:SWIT ON")
scope.write("CHAN1:COUP DC")
scope.write("CHAN2:COUP DC")
scope.write("CHAN3:SWIT OFF")
scope.write("CHAN1:SCAL 0.1")
scope.write("CHAN2:SCAL 0.1")
scope.write("CHAN1:OFFS 0")
scope.write("CHAN2:OFFS 0")
scope.write("CHAN1:VIS ON")
scope.write("CHAN2:VIS ON")
scope.write("TIM:SCAL .1e-3")
scope.write("ACQ:MDEP 10k")

# We will use Ch4 of the scope connected to the sync out of the 
# generator for triggering
scope.write("CHAN4:SWIT ON")
scope.write("CHAN4:SCAL 2")
scope.write("TRIG:EDGE:SOUR C4")
scope.write("TRIG:EDGE:LEV 1")
scope.write("CHAN4:VIS OFF")
scope.write("TRIG:RUN")

def setup_frequency_sweep(start_frequency, stop_frequency, sweep_time=1):
    # Ok, scope time base on screen goes in increments 1, 2, 5, 10...    
    mantissa = float(("%e" %  sweep_time).split("e")[0])
    exponent = float(("%e" %  sweep_time).split("e")[1])
    allowed = np.array([1, 2, 5, 10])
    idx = np.abs(allowed - mantissa).argmin()
    mantissa = allowed[idx]
    sweep_time = float("%fe%d" % (mantissa, exponent))
    print("Picking sweep time %e based on nearest allowed division of scope display" % sweep_time)
    
    # Configurting sweep, page 26 of manual of SDG1000 
    gen.write("C1:SWWV STATE,ON")
    gen.write("C1:SWWV TIME,%f" % sweep_time)
    gen.write("C1:SWWV STOP,%f" % stop_frequency)
    gen.write("C1:SWWV START,%f" % start_frequency)
    
    # This will automatically set up the time base of the scope in a good way :)
    scope.write("TIM:SCAL %f" % (sweep_time/10)) # The display has 10 "divisions"...
    scope.write("TIM:DEL %f" % (sweep_time/2)) # Set horizontal position to show full sweep on screen
    print("Setting memory depth to 1 million points") # Will make trace downloding slow but gives best data
    scope.write("ACQ:MDEP 1M")

# this assumes your traces are acquired as above with the start and stop frequencies at start and end of trace
# We may need to 
def make_response_function(v_out, v_in, f_start, f_stop):
    f = np.linspace(f_start, f_stop, len(v_in))
    v_out_t = hilbert(v_in)
    v_in_t = hilbert(v_in)
    R = v_out_t / v_in_t
    return f, R

First, we configure the sweep:

In [ ]:
f1 = 1e3  # You can change f1 and f2 to achieve different frequency ranges
f2 = 1e6
setup_frequency_sweep(f1, f2, sweep_time = 1)

On the screen of the scope, we can in principle now already visually see the amplitude reponse. We may need to adjust the vertical scales of the scope, and also you may need to reduce or increase the amplitude of the generator Ch1 in to prevent your amplifier from clipping / saturating. 

In [ ]:
scope.get_screenshot()

Once you think it looks good, you can grab the traces and calculate the response function:

In [ ]:
scope.write("TRIG:STOP")
t,v_in = scope.get_trace(1, npoints='all')
t,v_out = scope.get_trace(2, npoints='all')
scope.write("TRIG:RUN")

f, A = make_response_function(v_out, v_in, f1, f2)

Gv = np.abs(A)
dB = 20*np.log10(Gv)
phase = np.angle(A)/np.pi*180

And then we can check out what it looks like!

In [ ]:
plt.figure(figsize=(16,4))
plt.subplot(131)
plt.plot(f, Gv)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Voltage Gain")
plt.subplot(132)
plt.plot(f, Gv)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Gain (dB)") # In dB, there is no difference between voltage gain and power gain...
plt.subplot(133)
plt.plot(f, phase)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplification Phase (degrees)")

# Level 2 work

Add your own code and code cells below for your level 2 work. 

In [ ]:
# Your level 2 code. Take screenshots for your logbook.